In [2]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [2]:
csv_path = 'population_data/Bangladesh_Upzilla_List_With_Status_and_Population.csv'
print(f"csv: {csv_path}")
df_pop = pd.read_csv(csv_path)

csv: population_data/Bangladesh_Upzilla_List_With_Status_and_Population.csv


In [3]:
pop_col = 'population2022' 
if df_pop[pop_col].dtype == object:
    df_pop[pop_col] = df_pop[pop_col].astype(str).str.replace(',', '').astype(float)



In [4]:
import geopandas as gpd

shp_folder = 'data_for_adm03_bangladesh'
shp_files = [f for f in os.listdir(shp_folder) if f.endswith('.shp')]

if not shp_files:
    raise FileNotFoundError(f"No .shp file found in {shp_folder}")

shp_path = os.path.join(shp_folder, shp_files[0])
print(f"shp file: {shp_path}")
gdf_boundaries = gpd.read_file(shp_path)

shp file: data_for_adm03_bangladesh\bgd_admbnda_adm3_bbs_20180410.shp


In [5]:
gdf_merged = gdf_boundaries.merge(
    df_pop, 
    on=['ADM3_EN', 'ADM2_EN'],
    how='left'
)
gdf_merged[pop_col] = gdf_merged[pop_col].fillna(0)
#simplifying 
gdf_merged['geometry'] = gdf_merged.geometry.simplify(tolerance=0.001, preserve_topology=True)
final_fc = geemap.geopandas_to_ee(gdf_merged)

In [6]:
pop_image = final_fc.reduceToImage(
    properties=[pop_col],
    reducer=ee.Reducer.first()
)

In [7]:
vis_params = {
    'min': 0,
    'max': 1000000, 
    'palette': ['FFFFFF', 'FFCCCC', 'FF6666', 'FF0000', '8B0000'] 
}

m = geemap.Map()
m.setCenter(90.35, 23.68, 7)
m.set_options('HYBRID')

m.addLayer(pop_image, vis_params, 'population 2022 density')
borders = ee.Image().paint(final_fc, 0, 1)
m.addLayer(borders, {'palette': '000000', 'opacity': 0.3}, 'Borders')
m.addLayer(final_fc, {'color': '00000000'}, 'data', True, 0.5)
legend_dict = {
    "High, >1M": "#8B0000",
    "Medium": "#FF0000",
    "Low": "#FFCCCC",
    "Zero or No Data": "#FFFFFF"
}

m.add_legend(title="Population Density", legend_dict=legend_dict, position='bottomright')
m

Map(center=[23.68, 90.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…